## PALIGATU AI

Nama anggota

- Muhammad Dafi Cahyady
- Faishal Dwi Rijaya
- Rafghan Fahrezi
- Nadia Zunairah
- Dwi Nova Wijaya Santoso

> Paligatu AI kami buat karena banyak warga sekolah masih kesulitan memilah sampah. Banyak siswa tidak menyadari perbedaan antara sampah organik, anorganik, dan B3, sehingga mereka sering tercampur dan salah buang. Namun, kesalahan menangani sampah B3 seperti baterai bekas merupakan bahaya bagi lingkungan dan kesehatan. Akibatnya, kami mengembangkan PaligatuAI, sebuah aplikasi berbasis kecerdasan buatan, untuk membantu siswa dan guru mengidentifikasi kategori sampah dengan hanya memotretnya, serta memberikan rekomendasi untuk tindakan yang tepat. Aplikasi ini juga berfungsi sebagai media untuk mengajarkan siswa dan guru memilah sampah sejak dini.

### 1 Install Library

In [ ]:
!pip install -q torch torchvision gradio scikit-fuzzy scikit-learn kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 20.9 MB/s eta 0:00:00


## 2 Import Library

In [ ]:
import os, shutil, numpy as np, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import gradio as gr
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ["organik", "anorganik", "b3"]
MODEL_PATH = "/content/best_model.pth"

# folder dibaca urut abjad sama ImageFolder, jadi anorganik=0, b3=1, organik=2
idx_to_class = {i: c for i, c in enumerate(sorted(CLASSES))}

print("Device:", device)
print("Mapping:", idx_to_class)

Device: cuda
Mapping: {0: 'anorganik', 1: 'b3', 2: 'organik'}


## 3 Download dan Load Datasets dari kaggle

In [ ]:
from google.colab import userdata
import os

os.environ["KAGGLE_USERNAME"] = userdata.get("Username")
os.environ["KAGGLE_KEY"] = userdata.get("API31")

!kaggle datasets download -d mostafaabla/garbage-classification -p /content --unzip -q

def cari_raw_dir(base="/content"):
    for d in os.listdir(base):
        p = os.path.join(base, d)
        if os.path.isdir(p) and "battery" in [x.lower() for x in os.listdir(p)]:
            return p
    return "/content/garbage_classification"

RAW_DIR = cari_raw_dir()
print("RAW_DIR:", RAW_DIR)
print("kelas asli:", sorted(os.listdir(RAW_DIR)))

Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0
RAW_DIR: /content/garbage_classification
kelas asli: ['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']


In [ ]:
# masukin baterai dari dataset lain ke folder battery, biar ikut keremap jadi b3
EXTRA_B3_DATASETS = [
    "sumn2u/garbage-classification-v2",
    "akshat103/e-waste-image-dataset",
]
EXTRA_DIR = "/content/extra_b3"
os.makedirs(EXTRA_DIR, exist_ok=True)
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def cari_folder_baterai(root):
    hasil = []
    for dirpath, dirnames, filenames in os.walk(root):
        if "batter" in os.path.basename(dirpath).lower():
            hasil.append(dirpath)
    return hasil

battery_dir = os.path.join(RAW_DIR, "battery")
os.makedirs(battery_dir, exist_ok=True)

total = 0
for ds in EXTRA_B3_DATASETS:
    nama = ds.split("/")[-1]
    dl_dir = os.path.join(EXTRA_DIR, nama)
    if not os.path.exists(dl_dir):
        os.makedirs(dl_dir, exist_ok=True)
        !kaggle datasets download -d {ds} -p {dl_dir} --unzip -q
    for bdir in cari_folder_baterai(dl_dir):
        for f in os.listdir(bdir):
            if f.lower().endswith(IMG_EXT):
                src = os.path.join(bdir, f)
                if os.path.isfile(src):
                    shutil.copy(src, os.path.join(battery_dir, "extra_" + nama + "_" + f))
                    total += 1

print("baterai tambahan:", total)
print("total battery sekarang:", len(os.listdir(battery_dir)))

Dataset URL: https://www.kaggle.com/datasets/akshat103/e-waste-image-dataset
License(s): apache-2.0
baterai tambahan: 2568
total battery sekarang: 2001


In [ ]:
EXTRA_DIR = "/content/extra_b3"
os.makedirs(EXTRA_DIR, exist_ok=True)
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

DATASET_BATERAI = "sumn2u/garbage-classification-v2"
DATASET_EWASTE  = "akshat103/e-waste-image-dataset"

def download_dataset(nama_ds):
    folder = os.path.join(EXTRA_DIR, nama_ds.split("/")[-1])
    if not os.path.exists(folder):
        os.makedirs(folder, exist_ok=True)
        !kaggle datasets download -d {nama_ds} -p {folder} --unzip -q
    return folder

B3_DIR = os.path.join(RAW_DIR, "ewaste")
os.makedirs(B3_DIR, exist_ok=True)

nomor = 0

folder1 = download_dataset(DATASET_BATERAI)
for dirpath, dirnames, filenames in os.walk(folder1):
    nama_folder = os.path.basename(dirpath).lower()
    if "batter" in nama_folder:
        for f in filenames:
            if f.lower().endswith(IMG_EXT):
                src = os.path.join(dirpath, f)
                if os.path.isfile(src):
                    nomor += 1
                    ext = os.path.splitext(f)[1].lower()
                    shutil.copy(src, os.path.join(B3_DIR, f"b3_{nomor}{ext}"))

folder2 = download_dataset(DATASET_EWASTE)
for dirpath, dirnames, filenames in os.walk(folder2):
    for f in filenames:
        if f.lower().endswith(IMG_EXT):
            src = os.path.join(dirpath, f)
            if os.path.isfile(src):
                nomor += 1
                ext = os.path.splitext(f)[1].lower()
                shutil.copy(src, os.path.join(B3_DIR, f"b3_{nomor}{ext}"))

print("total gambar e-waste/B3 tambahan:", len(os.listdir(B3_DIR)))


total gambar e-waste/B3 tambahan: 5268


## 4 Remap

In [ ]:
REMAP = {
    "biological": "organik",

    "paper": "anorganik", "cardboard": "anorganik", "plastic": "anorganik",
    "metal": "anorganik", "white-glass": "anorganik", "green-glass": "anorganik",
    "brown-glass": "anorganik",

    "battery": "b3",
    "ewaste":  "b3",
}
OUT_DIR = "/content/dataset"
if os.path.exists(OUT_DIR): shutil.rmtree(OUT_DIR)
for c in CLASSES: os.makedirs(os.path.join(OUT_DIR, c), exist_ok=True)

for src, dst in REMAP.items():
    sdir = os.path.join(RAW_DIR, src)
    if not os.path.isdir(sdir): continue
    for f in os.listdir(sdir):
        shutil.copy(os.path.join(sdir, f), os.path.join(OUT_DIR, dst, f"{src}_{f}"))

for c in CLASSES:
    print(c, ":", len(os.listdir(os.path.join(OUT_DIR, c))), "gambar")


organik : 985 gambar
anorganik : 5586 gambar
b3 : 7269 gambar


## 5 Buat data Loader

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.GaussianBlur(3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# validasi
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# bagi data: 80% buat latih, 20% buat validasi
full = datasets.ImageFolder(OUT_DIR)
n_val = int(0.2 * len(full))
n_train = len(full) - n_val
g = torch.Generator().manual_seed(42)
train_set, val_set = torch.utils.data.random_split(full, [n_train, n_val], generator=g)

# augmentasi data
train_set.dataset.transform = train_tf
val_set = torch.utils.data.Subset(datasets.ImageFolder(OUT_DIR, transform=val_tf), val_set.indices)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)

counts = torch.tensor([len(os.listdir(os.path.join(OUT_DIR, c))) for c in full.classes], dtype=torch.float)
class_weights = (counts.sum() / (len(counts) * counts)).to(device)
print("Mapping kelas:", full.class_to_idx)
print("Jumlah per kelas:", counts.tolist())
print("Class weights:", class_weights.tolist())


Mapping kelas: {'anorganik': 0, 'b3': 1, 'organik': 2}
Jumlah per kelas: [5586.0, 7269.0, 985.0]
Class weights: [0.8258742094039917, 0.6346585750579834, 4.683587074279785]


## 6 Load Model EfficientNet-B0

In [ ]:
def build_model(unfreeze_last=True):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for p in m.features.parameters():
        p.requires_grad = False
    if unfreeze_last:
        for p in m.features[-2:].parameters():
            p.requires_grad = True
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_f, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, len(CLASSES)),
    )
    return m.to(device)

model = build_model()
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 132MB/s]


trainable params: 1458099


## 7 Training Data

In [ ]:
import torch
print("CUDA tersedia:", torch.cuda.is_available())
print("Device model:", next(model.parameters()).device)

CUDA tersedia: True
Device model: cuda:0


In [ ]:
EPOCHS = 12
criterion = nn.CrossEntropyLoss(weight=class_weights)
# backbone LR kecil, classifier LR gede
optimizer = torch.optim.Adam([
    {"params": model.features[-2:].parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(),    "lr": 1e-3},
])

best_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    acc = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} - val_acc: {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), MODEL_PATH)
        print("  model disimpan, acc:", round(acc, 4))

print("best val acc:", best_acc)
print("jangan lupa download best_model.pth dari panel Files")

Epoch 1/12 - val_acc: 0.9621
  model disimpan, acc: 0.9621
Epoch 2/12 - val_acc: 0.9729
  model disimpan, acc: 0.9729
Epoch 3/12 - val_acc: 0.9776
  model disimpan, acc: 0.9776
Epoch 4/12 - val_acc: 0.9762
Epoch 5/12 - val_acc: 0.9801
  model disimpan, acc: 0.9801
Epoch 6/12 - val_acc: 0.9794
Epoch 7/12 - val_acc: 0.9841
  model disimpan, acc: 0.9841
Epoch 8/12 - val_acc: 0.9823
Epoch 9/12 - val_acc: 0.9798
Epoch 10/12 - val_acc: 0.9816
Epoch 11/12 - val_acc: 0.9841
Epoch 12/12 - val_acc: 0.9790
best val acc: 0.9841040462427746
jangan lupa download best_model.pth dari panel Files


## 8 Evaluasi Akurasi

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        y_pred += model(x).argmax(1).cpu().tolist()
        y_true += y.tolist()

idx_to_class = {v: k for k, v in full.class_to_idx.items()}
target_names = [idx_to_class[i] for i in range(len(CLASSES))]
print(classification_report(y_true, y_pred, target_names=target_names, digits=3))

              precision    recall  f1-score   support

   anorganik      0.979     0.984     0.981      1178
          b3      0.989     0.983     0.986      1393
     organik      0.980     0.990     0.985       197

    accuracy                          0.984      2768
   macro avg      0.983     0.986     0.984      2768
weighted avg      0.984     0.984     0.984      2768



##  9 Fuzzy Logic

In [ ]:
conf = ctrl.Antecedent(np.arange(0, 101, 1), "conf")
trust = ctrl.Consequent(np.arange(0, 101, 1), "trust")

conf["rendah"]  = fuzz.trapmf(conf.universe, [0, 0, 40, 55])
conf["sedang"]  = fuzz.trimf(conf.universe, [50, 62, 78])
conf["tinggi"]  = fuzz.trapmf(conf.universe, [72, 85, 100, 100])
trust["rendah"] = fuzz.trimf(trust.universe, [0, 0, 50])
trust["sedang"] = fuzz.trimf(trust.universe, [40, 60, 80])
trust["tinggi"] = fuzz.trimf(trust.universe, [70, 100, 100])

rules = [
    ctrl.Rule(conf["rendah"], trust["rendah"]),
    ctrl.Rule(conf["sedang"], trust["sedang"]),
    ctrl.Rule(conf["tinggi"], trust["tinggi"]),
]
fuzzy_sim = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))

def cek_keyakinan(confidence_pct):
    fuzzy_sim.input["conf"] = confidence_pct
    fuzzy_sim.compute()
    skor = fuzzy_sim.output["trust"]
    if confidence_pct < 50:
        return "Tidak Yakin", "Foto kurang jelas, coba foto ulang lebih deket sama terang.", skor
    elif confidence_pct < 75:
        return "Cukup Yakin", "Hasil sementara, dicek manual dulu sebelum dibuang.", skor
    else:
        return "Yakin", "Hasil bisa langsung dipakai.", skor

print(cek_keyakinan(90)); print(cek_keyakinan(60)); print(cek_keyakinan(35))

('Yakin', 'Hasil bisa langsung dipakai.', np.float64(90.00000000000003))
('Cukup Yakin', 'Hasil sementara, dicek manual dulu sebelum dibuang.', np.float64(60.00000000000002))
('Tidak Yakin', 'Foto kurang jelas, coba foto ulang lebih deket sama terang.', np.float64(16.666666666666664))


## 10 Prediksi

In [ ]:
SARAN = {
    "organik":   "Buang ke tempat sampah hijau, bisa dijadiin kompos.",
    "anorganik": "Buang ke tempat sampah biru, bisa didaur ulang / jual ke bank sampah.",
    "b3":        "Jangan dibuang sembarangan, serahin ke TPS B3 / dinas lingkungan.",
}

# kalau lagi demo dan modelnya belum ada, minta upload dulu
if not os.path.exists(MODEL_PATH):
    from google.colab import files
    print("best_model.pth belum ada, upload dulu:")
    up = files.upload()
    fname = list(up.keys())[0]
    if fname != "best_model.pth":
        shutil.move(fname, MODEL_PATH)

if "model" not in globals():
    model = build_model()
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

def predict(img):
    if img is None:
        return "Tidak ada gambar."
    x = val_tf(img.convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.softmax(model(x), 1)[0].cpu().numpy()
    idx = int(prob.argmax())
    label = idx_to_class[idx] if "idx_to_class" in globals() else CLASSES[idx]
    conf_pct = float(prob[idx] * 100)
    status, pesan, _ = cek_keyakinan(conf_pct)
    extra = "\n\nNote: B3 tetep perlu dicek manual." if label == "b3" else ""
    return (f"Kategori : {label.upper()}\n"
            f"Confidence : {conf_pct:.1f}%\n"
            f"Keyakinan : {status}\n"
            f"Catatan : {pesan}\n\n"
            f"Saran : {SARAN[label]}{extra}")

## 11 Demo Apps

In [ ]:
js_rear_cam = """
() => {
    if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia) return;
    const orig = navigator.mediaDevices.getUserMedia.bind(navigator.mediaDevices);
    navigator.mediaDevices.getUserMedia = (constraints) => {
        if (constraints && constraints.video) {
            if (constraints.video === true) constraints.video = {};
            constraints.video.facingMode = { ideal: "environment" };
        }
        return orig(constraints);
    };
}
"""

with gr.Blocks(title="PaligatuAI") as demo:
    gr.Markdown("#PaligatuAI — Klasifikasi Sampah Cerdas\nFoto, Pilah, Peduli. Unggah atau ambil foto sampah, AI memilah ke Organik, Anorganik, atau B3.")
    with gr.Row():
        inp = gr.Image(type="pil", sources=["upload", "webcam"], label="Foto Sampah")
        out = gr.Textbox(label="Hasil Klasifikasi", lines=8)
    inp.change(predict, inputs=inp, outputs=out)
    demo.load(None, None, None, js=js_rear_cam)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://588ea5244eded57aab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
